# Project 010: Gradient Boosting Algorithm Implementation & Studio

**Category:** Machine Learning Basic Projects  
**Domain:** Foundational Machine Learning & Ensemble Methods  
**Algorithm:** Gradient Tree Boosting (Friedman Additive Modeling)  

---

### 📖 Overview & Theoretical Core
Gradient Boosting is an additive ensemble technique that constructs a strong predictive model by sequentially fitting base learners (shallow regression trees) to the negative gradient of a differentiable loss function (pseudo-residuals).

The fundamental additive update formula:
$$F_M(x) = F_0(x) + \sum_{m=1}^M \eta \cdot h_m(x)$$

Where:
- $F_0(x)$ is the initial optimal constant prediction (e.g. log-odds for binary classification)
- $h_m(x)$ is the weak learner trained at stage $m$ to fit pseudo-residuals $r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}$
- $\eta \in (0, 1]$ is the **shrinkage factor (learning rate)** preventing rapid overfitting
- $M$ is the number of boosting stages (`n_estimators`)

---

### 🎯 Learning Objectives
1. Train and evaluate a Gradient Tree Boosting Classifier on real and synthetic benchmarks
2. Track stage-by-stage deviance (loss) convergence and identify the optimal early-stopping point
3. Investigate the shrinkage tradeoff: learning rate ($\eta$) vs number of trees ($M$)
4. Extract Mean Decrease in Impurity (MDI) feature importances and inspect individual tree architectures

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataset import load_real_dataset, generate_2d_manifold
from gbm_engine import (
    prepare_dataset_split,
    train_gradient_boosting,
    compute_staged_metrics,
    evaluate_model,
    extract_feature_importances,
    extract_forest_diagnostics,
    run_cross_validation
)

# Visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
print("✓ Core modules successfully imported.")

## 1. Load & Inspect Benchmark Dataset (Credit Default & Risk)

In [ ]:
df, feature_cols, target_col, target_names = load_real_dataset("credit")
print(f"Dataset Shape: {df.shape}")
print(f"Target Classes: {target_names}")
print("\nFirst 5 Records:")
display(df.head())

# Class distribution
print("\nClass Distribution:")
print(df['target_label'].value_counts())

## 2. Train Gradient Boosting Classifier & Compute Staged Deviance

In [ ]:
X_train, X_test, y_train, y_test = prepare_dataset_split(
    df=df,
    feature_cols=feature_cols,
    target_col=target_col,
    test_size=0.25,
    random_state=42
)

# Train baseline model
n_estimators = 100
learning_rate = 0.08
model = train_gradient_boosting(
    X_train=X_train,
    y_train=y_train,
    n_estimators=n_estimators,
    learning_rate=learning_rate,
    max_depth=3,
    subsample=1.0,
    random_state=42
)

# Compute staged deviance progression
staged = compute_staged_metrics(model, X_train, y_train, X_test, y_test)
print(f"Optimal Early Stopping Stage: Tree #{staged['best_stage']}")
print(f"Min Test Deviance (Loss):    {staged['min_test_deviance']:.4f}")
print(f"Final Test Deviance:         {staged['test_deviance'][-1]:.4f}")

## 3. Visualize Stage-by-Stage Deviance (Loss Trajectory)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(staged["iterations"], staged["train_deviance"], label="Training Deviance", color="#10B981", lw=2.2)
plt.plot(staged["iterations"], staged["test_deviance"], label="Testing Deviance", color="#6366F1", lw=2.2)
plt.axvline(x=staged["best_stage"], color="#EF4444", linestyle="--", label=f"Optimal Early Stop (#{staged['best_stage']})")
plt.scatter([staged["best_stage"]], [staged["min_test_deviance"]], color="#EF4444", s=80, zorder=5)

plt.title("Stage-by-Stage Deviance (Loss) Trajectory in Gradient Boosting", fontsize=13, fontweight="bold")
plt.xlabel("Boosting Iterations (Number of Trees)")
plt.ylabel("Bernoulli Deviance (Log-Loss)")
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

## 4. Mean Decrease in Impurity (MDI) Feature Importances

In [ ]:
imp_df = extract_feature_importances(model, feature_cols)

plt.figure(figsize=(10, 5))
plt.barh(imp_df["Feature"][::-1], imp_df["Percentage"][::-1], color="#3B82F6", edgecolor="black", alpha=0.85)
plt.title("Gradient Boosting Feature Importance Ranking (MDI %)", fontsize=13, fontweight="bold")
plt.xlabel("Relative Importance (%)")
plt.tight_layout()
plt.show()

display(imp_df)

## 5. Comprehensive Model Evaluation & Cross-Validation

In [ ]:
eval_res = evaluate_model(model, X_test, y_test, target_names=target_names)
print("=" * 50)
print(f"Test Accuracy:   {eval_res['accuracy'] * 100:.2f}%")
print(f"ROC-AUC Score:   {eval_res['roc_auc']:.4f}")
print(f"Precision:       {eval_res['precision'] * 100:.2f}%")
print(f"Recall:          {eval_res['recall'] * 100:.2f}%")
print(f"F1-Score:        {eval_res['f1_score'] * 100:.2f}%")
print(f"Log-Loss:        {eval_res['log_loss']:.4f}")
print("=" * 50)

# 5-Fold Stratified Cross-Validation
cv_res = run_cross_validation(
    X=df[feature_cols].to_numpy(),
    y=df[target_col].to_numpy(),
    n_estimators=100,
    learning_rate=0.08,
    cv_folds=5
)
print(f"\n5-Fold CV Mean Accuracy: {cv_res['accuracy_mean'] * 100:.2f}% (± {cv_res['accuracy_std'] * 100:.2f}%)")
print(f"5-Fold CV Mean ROC-AUC:  {cv_res['roc_auc_mean']:.4f} (± {cv_res['roc_auc_std']:.4f})")